# Identify analyses to rerun

Build the list of country–analysis pairs to submit via `scripts/massive/jtrauer/launch.sh`, and write it to `data/config/rerun_pairs.json`.

Two cases are included:

1. **Missing** — requested pairs with no usable output anywhere in `FULL_RUN` (after accounting for mobility skips).
2. **Superseded** — the most recent gap-fill job (`FULL_RUN[0]`, currently `60126550`) contains a directory for the pair but it is not usable, while an older job still supplies the copy that `get_analysis_paths` selects. Example: Malaysia `oxcgrt_floored` (failed 2-chain rerun; analysis still falls back to pre–H1-removal outputs in `59597639`).

**Usable** (`analysis_output_status` returns `"usable"`): non-empty `updates.h5`, `spaghetti.h5`, and `idata_filtered.nc` ≥ `MIN_IDATA_BYTES`.

**Skipped**: mobility analyses logged as `{analysis} data not available`, or with no matching file in `data/mobility/`. These are omitted from the rerun list.

**Job order**: `FULL_RUN` in `constants.py` — first usable copy wins. Prepend a new job ID there after a remote run if its outputs should take precedence.

When both missing and superseded lists are empty, no targeted rerun is needed.


In [ ]:
import json

import pandas as pd

from emu_renewal.constants import ANALYSIS_TYPES, DATA_PATH, OUTPUTS_PATH, FULL_RUN
from emu_renewal.run import get_analyses_for_country
from emu_renewal.utils import analysis_output_status

In [ ]:
job_ids = FULL_RUN
countries = json.load(open(DATA_PATH / "config/oxcgrt_included.json"))

In [ ]:
def classify_analysis(iso3, analysis, run_path, log_text):
    analysis_path = run_path / iso3 / analysis
    if log_text and f"{analysis} data not available" in log_text:
        return "skipped"
    return analysis_output_status(analysis_path)


def classify_job(run_id):
    run_path = OUTPUTS_PATH / run_id
    status = pd.DataFrame(index=countries, columns=ANALYSIS_TYPES)
    for iso3 in countries:
        log_path = run_path / iso3 / "run.log"
        log_text = log_path.read_text() if log_path.exists() else None
        requested_types = get_analyses_for_country(iso3)
        for analysis in ANALYSIS_TYPES:
            if analysis not in requested_types:
                status.loc[iso3, analysis] = "not requested"
            else:
                status.loc[iso3, analysis] = classify_analysis(iso3, analysis, run_path, log_text)
    return status


statuses = {job: classify_job(job) for job in job_ids}
pd.concat(
    {job: s.apply(pd.Series.value_counts).fillna(0).astype(int) for job, s in statuses.items()},
    axis=1,
).fillna(0).astype(int)

In [ ]:
MOB_SCALER_FILES = {
    "g_mob": "gmob_data.csv",
    "fb_visited_mob": "fbmob_data.csv",
    "fb_singletile_mob": "fbsingletile_data.csv",
}


def has_mobility_scaler(iso3, analysis):
    filename = MOB_SCALER_FILES.get(analysis)
    if filename is None:
        return True
    return (DATA_PATH / "mobility" / f"{iso3}_{filename}").exists()


def pairs_from_mask(mask):
    stacked = mask.stack()
    return list(stacked[stacked].index)


requested = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
for iso3 in countries:
    requested.loc[iso3, get_analyses_for_country(iso3)] = True

skipped = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
claimed = pd.DataFrame(False, index=countries, columns=ANALYSIS_TYPES)
summary = {"requested": int(requested.to_numpy().sum())}
for job in job_ids:
    skipped = skipped | (statuses[job] == "skipped")
    usable = statuses[job] == "usable"
    summary[f"available from {job}"] = int((requested & usable & ~claimed).to_numpy().sum())
    claimed = claimed | usable

no_scaler = pd.DataFrame(
    [
        [has_mobility_scaler(iso3, analysis) for analysis in ANALYSIS_TYPES]
        for iso3 in countries
    ],
    index=countries,
    columns=ANALYSIS_TYPES,
)

summary["skipped (logged)"] = int((requested & skipped).to_numpy().sum())
summary["no mobility scaler CSV"] = int((requested & ~no_scaler).to_numpy().sum())
need = requested & ~skipped & ~claimed & no_scaler
missing_pairs = sorted(pairs_from_mask(need))
summary["missing"] = len(missing_pairs)


def find_superseded_pairs(recent_job_id: str) -> list[tuple[str, str]]:
    """Pairs where the newest gap-fill job tried but failed, and an older job is still selected."""
    pairs = []
    for iso3 in countries:
        for analysis in get_analyses_for_country(iso3):
            recent_path = OUTPUTS_PATH / recent_job_id / iso3 / analysis
            if not recent_path.is_dir():
                continue
            if analysis_output_status(recent_path) == "usable":
                continue
            for job in job_ids[1:]:
                older_path = OUTPUTS_PATH / job / iso3 / analysis
                if analysis_output_status(older_path) == "usable":
                    pairs.append((iso3, analysis))
                    break
    return sorted(pairs)


superseded_pairs = find_superseded_pairs(job_ids[0])
summary["superseded"] = len(superseded_pairs)
rerun_pairs = sorted(set(missing_pairs) | set(superseded_pairs))
summary["total to rerun"] = len(rerun_pairs)
pd.Series(summary)

In [ ]:
def rerun_detail(iso3, analysis, reason):
    row = {
        "iso3": iso3,
        "analysis": analysis,
        "reason": reason,
    }
    row.update(
        {job: analysis_output_status(OUTPUTS_PATH / job / iso3 / analysis) for job in job_ids}
    )
    return row


if missing_pairs:
    print("Missing (no usable output in FULL_RUN):")
    display(pd.DataFrame([rerun_detail(iso3, analysis, "missing") for iso3, analysis in missing_pairs]))
else:
    print("No missing pairs.")

if superseded_pairs:
    print(f"Superseded (failed attempt in {job_ids[0]}, older job still selected):")
    display(pd.DataFrame([rerun_detail(iso3, analysis, "superseded") for iso3, analysis in superseded_pairs]))
else:
    print("No superseded pairs.")

if rerun_pairs:
    print(f"\nTotal to write to rerun_pairs.json ({len(rerun_pairs)}):")
    display(pd.DataFrame(rerun_pairs, columns=["iso3", "analysis"]))
else:
    print("No reruns required")

In [ ]:
out_path = DATA_PATH / "config/rerun_pairs.json"
json.dump(rerun_pairs, open(out_path, "w"))
print(f"Wrote {len(rerun_pairs)} pair(s) to {out_path}")
rerun_pairs